# Proyecto: Data Stream Processor

## 1. Importación de Estructuras Base
En esta sección se importan los Tipos de Datos Abstractos (TDA) de Pila (`ArrayStack`) y Cola (`ArrayQueue`), 
junto con la excepción personalizada `Empty`, provenientes del paquete `goodrich`.

- **ArrayQueue:** Administra la cola de registros pendientes bajo la política FIFO (First In, First Out).
- **ArrayStack:** Almacena el historial de modificaciones bajo la política LIFO (Last In, First Out) para permitir la funcionalidad de deshacer (`undo`).
- **Empty:** Excepción personalizada que se lanza cuando se intenta acceder o retirar elementos de una pila o cola vacía.

In [2]:
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue
from goodrich.exceptions import Empty

## 2. Definición de la Clase `DataProcessor`

La clase `DataProcessor` actúa como el motor central del sistema. Mantiene tres componentes principales en su estado interno:

1. **`_queue` (`ArrayQueue`):** Registro de datos en espera de ser procesados (Orden FIFO).
2. **`_stack` (`ArrayStack`):** Registro de historial de cambios para permitir operaciones `undo` (Orden LIFO).
3. **`_state` (`list`):** Lista que contiene los registros procesados en su estado actual `(sensor, variable, value)`.

In [4]:
class DataProcessor:
    def __init__(self):
        """
        Inicializa el procesador de flujo de datos con:
        - Una cola vacía para registros pendientes (FIFO).
        - Una pila vacía para el historial de cambios (LIFO).
        - Una lista vacía para mantener el estado actual de los datos.
        """
        self._queue = ArrayQueue()
        self._stack = ArrayStack()
        self._state = []

    def pending(self) -> int:
        """
        Devuelve el número de registros pendientes por procesar en la cola.
        """
        return len(self._queue)

    def add(self, record):
        """
        Agrega un registro a la cola de procesamiento pendiente.
        
        Requisitos del registro:
        - Debe ser una tupla o lista de exactamente 3 elementos: (sensor, variable, value).
        - 'value' debe ser un tipo numérico (int o float).
        
        Si el formato o tipo de dato es incorrecto, se lanza un ValueError.
        """
        # 1. Validación de estructura: debe ser una tupla/lista de 3 elementos
        if not isinstance(record, (tuple, list)) or len(record) != 3:
            raise ValueError("El registro debe ser una tupla o lista de exactamente 3 elementos: (sensor, variable, value)")
        
        sensor, variable, value = record
        
        # 2. Validación del tipo de valor: debe ser un entero o flotante
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError("El valor del registro (tercer elemento) debe ser numérico (int o float)")

        # 3. Encolar el registro validado sin procesarlo
        self._queue.enqueue((sensor, variable, value))

## 3. Procesamiento de Registros y Consulta de Estado

En esta sección se implementa la lógica para consumir los registros de la cola y actualizar el estado actual del sistema:

### Lógica de `process_next()`
1. **Desencolado (FIFO):** Extrae el primer registro disponible de la cola pendientes (`ArrayQueue`). Si la cola está vacía, eleva la excepción `Empty`.
2. **Gestión del Historial para Deshacer:** Antes de modificar el estado, busca si la clave única `(sensor, variable)` ya existe en la lista de estado actual (`self._state`):
   - **Si ya existía:** Se almacena en la pila (`ArrayStack`) una tupla `(sensor, variable, old_value)` conservando el valor previo.
   - **Si no existía (dato nuevo):** Se almacena en la pila `(sensor, variable, None)` para indicar que el estado anterior era "inexistente".
3. **Actualización de Estado:** Se sobrescribe el registro en la lista si existía previa coincidencia, o se añade al final en caso contrario.

### Lógica de `current_value(sensor, variable)`
- Recorre la lista de estado actual buscando la coincidencia del sensor y la variable requerida.
- Retorna el valor actual si existe, o eleva la excepción `KeyError` si nunca se ha procesado una lectura para esa combinación.

In [ ]:
class DataProcessor:
    def __init__(self):
        """
        Inicializa el procesador de flujo de datos con:
        - Una cola vacía para registros pendientes (FIFO).
        - Una pila vacía para el historial de cambios (LIFO).
        - Una lista vacía para mantener el estado actual de los datos.
        """
        self._queue = ArrayQueue()
        self._stack = ArrayStack()
        self._state = []

    def pending(self) -> int:
        """
        Devuelve el número de registros pendientes por procesar en la cola.
        """
        return len(self._queue)

   def add(self, record):
        """
        Agrega un registro (sensor, variable, valor) a la cola de pendientes.
        Lanza ValueError si el formato no tiene 3 elementos o el valor no es numérico.
        """
        if len(record) != 3:
            raise ValueError("El registro debe tener exactamente 3 elementos.")

        sensor = record[0]
        variable = record[1]
        value = record[2]

        if type(value) != int and type(value) != float:
            raise ValueError("El valor del registro debe ser numérico.")

        self._queue.enqueue((sensor, variable, value))
    def process_next(self):
        """
        Procesa el primer registro en cola (FIFO), actualiza el estado y guarda el historial.
        Lanza Empty si no hay registros pendientes en la cola.
        """
        # 1. Si la cola está vacía, lanzar la excepción correspondientes
        if self._queue.is_empty():
            raise Empty("No hay registros pendientes para procesar.")

        # 2. Extraer el registro pendiente de la cola
        record = self._queue.dequeue()
        sensor = record[0]
        variable = record[1]
        new_value = record[2]

        # 3. Buscar si la clave (sensor, variable) ya existe en la lista de estado
        posicion_encontrada = -1
        valor_anterior = None

        for i in range(len(self._state)):
            item_actual = self._state[i]
            if item_actual[0] == sensor and item_actual[1] == variable:
                posicion_encontrada = i
                valor_anterior = item_actual[2]
                break

        # 4. Guardar en la pila la notita del estado previo (para el undo)
        self._stack.push((sensor, variable, valor_anterior))

        # 5. Actualizar la lista de estado actual
        if posicion_encontrada != -1:
            self._state[posicion_encontrada] = (sensor, variable, new_value)
        else:
            self._state.append((sensor, variable, new_value))

        return record